In [8]:
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel
from scipy.stats import norm
from scipy.optimize import minimize
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)  # makes results reproducible


# ==============================
# LOAD DATA
# ==============================
X = np.load("initial_inputs.npy")
y = np.load("initial_outputs.npy").reshape(-1)

if X.ndim == 1:
    X = X.reshape(-1, 1)

N, D_total = X.shape

print("Samples:", N)
print("Total dimension available:", D_total)
print("-" * 50)


# ==============================
# BO FUNCTION
# ==============================
def propose_next_query(X, y):

    n, d = X.shape

    mins = X.min(axis=0)
    maxs = X.max(axis=0)
    bounds = list(zip(mins, maxs))

    kernel = (
        ConstantKernel(1.0) *
        RBF(length_scale=np.ones(d)) +
        WhiteKernel(noise_level=1e-6)
    )

    gp = GaussianProcessRegressor(
        kernel=kernel,
        normalize_y=True,
        n_restarts_optimizer=5
    )

    gp.fit(X, y)

    y_best = np.min(y)
    xi = 1e-3

    def EI(x):
        x = np.asarray(x).reshape(1, d)
        mu, sigma = gp.predict(x, return_std=True)

        mu = mu.item()
        sigma = max(sigma.item(), 1e-12)

        imp = y_best - mu - xi
        Z = imp / sigma

        return imp * norm.cdf(Z) + sigma * norm.pdf(Z)

    # Random global search
    candidates = np.random.uniform(mins, maxs, size=(20000, d))
    eis = np.array([EI(c) for c in candidates])
    x0 = candidates[np.argmax(eis)]

    # Local refinement
    def neg_ei(x):
        return -EI(x)

    best = x0
    best_val = EI(x0)

    for _ in range(15):
        x_init = np.random.uniform(mins, maxs)
        res = minimize(neg_ei, x_init, bounds=bounds, method="L-BFGS-B")

        if res.success and -res.fun > best_val:
            best_val = -res.fun
            best = res.x

    return best


# ==============================
# RUN FOR 2D–8D
# ==============================

results = {}

for D in range(2, min(9, D_total + 1)):

    print(f"\nRunning Bayesian Optimisation for {D}D")

    Xd = X[:, :D]
    next_point = propose_next_query(Xd, y)

    results[D] = next_point

    print("Next query point:")
    print(np.round(next_point, 6))

    np.save(f"next_query_point_{D}d.npy", next_point)

print("\nAll dimensions complete.")

print("\nSummary of proposed points:")
for D, point in results.items():
    print(f"{D}D → {np.round(point,6)}")

Samples: 40
Total dimension available: 8
--------------------------------------------------

Running Bayesian Optimisation for 2D
Next query point:
[0.984904 0.754637]

Running Bayesian Optimisation for 3D
Next query point:
[0.985945 0.403364 0.866361]

Running Bayesian Optimisation for 4D
Next query point:
[0.985945 0.97398  0.998885 0.558132]

Running Bayesian Optimisation for 5D
Next query point:
[0.960457 0.755783 0.883857 0.223819 0.986902]

Running Bayesian Optimisation for 6D
Next query point:
[0.917568 0.189894 0.937759 0.620063 0.590335 0.231388]

Running Bayesian Optimisation for 7D
Next query point:
[0.985526 0.034959 0.968613 0.899856 0.73534  0.054146 0.932533]

Running Bayesian Optimisation for 8D
Next query point:
[0.961259 0.821777 0.996315 0.902239 0.598238 0.194292 0.716328 0.491093]

All dimensions complete.

Summary of proposed points:
2D → [0.984904 0.754637]
3D → [0.985945 0.403364 0.866361]
4D → [0.985945 0.97398  0.998885 0.558132]
5D → [0.960457 0.755783 0.8838